# Week 6 — Spark Data Processing Assignment
**Objective:** Understand Spark architecture and perform efficient data processing using
transformations, filtering, schema handling, and optimized file formats (CSV vs Parquet).

**Dataset:** `data/source.csv` — a small retail/e-commerce style product dataset
(product_id, product_name, category, price, quantity, region, priority, status).

**Pipeline:** Read → Schema Validation → Clean Nulls → Filter → Transform → Write (CSV & Parquet)


## 1. Spark Session Creation
Create a `SparkSession` — the entry point to any Spark application. The **Driver** runs this code and creates a logical plan; the **Cluster Manager** (here, Spark's built-in *local* manager) allocates resources; **Executors** run the actual tasks.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, upper
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("Week6_Spark_Data_Processing") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version         :", spark.version)
print("Application Name      :", spark.sparkContext.appName)
print("Master                :", spark.sparkContext.master)
print("Default Parallelism   :", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/26 21:55:38 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/26 21:55:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/07/26 21:55:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version         : 4.2.0
Application Name      : Week6_Spark_Data_Processing
Master                : local[*]
Default Parallelism   : 1


## 2. Load Dataset
Read the CSV with an **explicit schema** (best practice for large/production datasets — avoids the extra read pass that `inferSchema=True` triggers) and, separately, with `inferSchema=True` as required by the assignment.

In [2]:
explicit_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("status", StringType(), True),
])

df = spark.read.csv("../data/source.csv", header=True, schema=explicit_schema)

df_inferred = spark.read.csv("../data/source.csv", header=True, inferSchema=True)

print("Row count:", df.count())
print("Columns  :", df.columns)

Row count: 20
Columns  : ['product_id', 'product_name', 'category', 'price', 'quantity', 'region', 'priority', 'status']


## 3. Schema Inspection

In [3]:
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)



In [4]:
df_inferred.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)



## 4. Data Cleaning — Handle Null Values
Check how many nulls exist in critical columns, then drop rows with a missing `price` (a required business field) and fill missing `quantity` with `0` instead of discarding those rows.

In [5]:
null_counts = df.select([
    (col(c).isNull().cast("int")).alias(c) for c in ["price", "quantity"]
]).groupBy().sum().collect()[0]

print("Null count in 'price'   :", null_counts['sum(price)'])
print("Null count in 'quantity':", null_counts['sum(quantity)'])

Null count in 'price'   : 2
Null count in 'quantity': 3
